# 🏆 E6 — 최종 모델: 실험에서 배운 것 전부 합치기

**T4 기준 전체 약 45~55분** (모델 다운로드 5분 + 학습 ~30분 + 평가 ~10분)

## E0~E5 실험 결과를 그대로 반영한 4가지 개선

| # | 개선 | 근거가 된 실험 결과 |
|---|---|---|
| 1 | **1.2B 모델 + 데이터 증량** (1,500 → 4,000개) | E5: 모델 크기가 유일하게 전 지표 일관 개선 |
| 2 | **질문 부분 손실 마스킹** (label=-100) | 답변 생성에만 학습을 집중 → 답변 품질 향상 |
| 3 | **회피성 답변 강화 필터** (학습 데이터만) | E5에도 회피 응답 6/10 잔존 → 데이터가 다음 병목 |
| 4 | **길이 보정 + 디코딩 선발전** (검증셋 20개로 선발) | E4: 디코딩이 겹침 지표에 영향. BLEU의 짧은 답 벌점도 고려 |

## 공정성 원칙 ⚖️

- **테스트셋 60개는 E0~E5와 완전히 동일** (같은 정제 파이프라인 + seed 42로 재구성)
- 디코딩 선발전은 테스트셋이 아닌 **별도 검증셋 20개**로 진행 (테스트셋 오염 방지)
- 강화 필터는 **학습 데이터에만** 적용 — 채점 기준은 그대로

> ⚙️ 새 런타임 권장 (T4 GPU). 이전 결과와 합쳐 보려면 `ablation_backup.zip`을 업로드하고 아래 복원 셀을 실행하세요.


In [ ]:
# (선택) 이전 실험 결과와 이어서 비교하려면: ablation_backup.zip을 좌측 파일탭에 업로드 후 실행
import os
if os.path.exists('ablation_backup.zip'):
    !unzip -qo ablation_backup.zip
    print('✅ 이전 결과 복원 완료')
else:
    print('backup zip 없음 — E6 단독으로 진행 (문제 없음)')

## 1. 환경 설정

In [ ]:
import time
RUN_START = time.time()

!pip install -q transformers==4.46.3 peft==0.15.2 accelerate==1.1.1 bitsandbytes

import torch, random, re, json as pyjson
import numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "⚠️ 런타임 → 런타임 유형 변경 → T4 GPU"
os.makedirs('results', exist_ok=True); os.makedirs('adapters', exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0))

## 2. 데이터 — 테스트셋은 그대로, 학습셋만 업그레이드

앞부분은 E0~E5와 **한 글자도 다르지 않은** 정제 파이프라인입니다 (같은 테스트셋을 재구성하기 위해).
그 뒤에 학습용 데이터에만 ① 회피성 답변 강화 필터 ② 증량(4,000개)을 적용합니다.

In [ ]:
if not os.path.exists('KoChatGPT'):
    !git clone -q https://github.com/airobotlab/KoChatGPT.git

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        txt = f.read().strip()
    try:
        data = pyjson.loads(txt)
        return data if isinstance(data, list) else [data]
    except pyjson.JSONDecodeError:
        return [pyjson.loads(l) for l in txt.splitlines() if l.strip()]

sft_raw = load_jsonl('KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl')

# ══ E0~E5와 동일한 정제 파이프라인 (테스트셋 재구성용 — 수정 금지) ══
refusal_pat = re.compile(r'(AI\s?언어\s?모델|인공지능 (챗봇|모델|어시스턴트))')

def clean_text(t):
    t = re.sub(r'https?://\S+', '', t)
    t = re.sub(r'(.)\1{4,}', r'\1\1\1', t)
    t = re.sub(r'\s+', ' ', t).strip()
    t = re.sub(r'^[\'\"`]+', '', t)
    return t.strip()

def is_bad(p, c):
    if len(p) < 2 or len(c) < 10 or len(c) > 400: return True
    if refusal_pat.search(c) and '할 수 없' in c and len(c) < 120: return True
    return False

seen, cleaned = set(), []
for d in sft_raw:
    p, c = clean_text(d['prompt']), clean_text(d['completion'])
    if (p, c) in seen or is_bad(p, c): continue
    seen.add((p, c)); cleaned.append({'prompt': p, 'completion': c})

AUG_TEMPLATES = [lambda p: f"다음 질문에 답해줘. {p}",
                 lambda p: f"{p} 알기 쉽게 설명해줘.",
                 lambda p: f"질문: {p}"]
random.shuffle(cleaned)
n_aug = int(len(cleaned) * 0.2)
augmented = [{'prompt': random.choice(AUG_TEMPLATES)(d['prompt']), 'completion': d['completion']}
             for d in random.sample(cleaned, n_aug)]
dataset_all = cleaned + augmented
random.shuffle(dataset_all)

EVAL_N = 60
test_set     = dataset_all[:EVAL_N]                 # ← E0~E5와 동일한 테스트셋
test_prompts = [d['prompt'] for d in test_set]
test_refs    = [d['completion'] for d in test_set]

# ══ 여기서부터 E6 전용 업그레이드 ══
val_set   = dataset_all[EVAL_N:EVAL_N+20]           # 디코딩 선발전용 검증셋 (학습 제외)
pool      = dataset_all[EVAL_N+20:]

# ① 회피성 답변 강화 필터: 답변 앞머리에 AI 자기언급이 나오는 데이터 제거 (학습셋에만!)
ai_open = re.compile(r'(AI|인공지능|언어\s?모델|어시스턴트|챗봇)')
strong  = [d for d in pool if not ai_open.search(d['completion'][:30])]
print(f"강화 필터: {len(pool):,}개 → {len(strong):,}개 (회피성 오프닝 {len(pool)-len(strong):,}개 제거)")

# ② 증량: 4,000개 학습
MAX_TRAIN = 4000
train_set = strong[:MAX_TRAIN]
ref_avg_len = int(np.mean([len(r) for r in test_refs]))
print(f"train {len(train_set)} / val {len(val_set)} / test {len(test_set)} | 참조 답변 평균 {ref_avg_len}자")

## 3. 1.2B 모델 + 8bit 양자화 + LoRA (r=16)

E5보다 어댑터 용량을 2배(r=8→16)로 늘려 큰 모델의 표현력을 더 활용합니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, TaskType, prepare_model_for_kbit_training

MODEL_NAME = 'skt/ko-gpt-trinity-1.2B-v0.5'
tok_kwargs = dict(bos_token='<s>', eos_token='</s>', unk_token='<unk>',
                  pad_token='<pad>', mask_token='<mask>')
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **tok_kwargs)
except Exception:
    tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_NAME, **tok_kwargs)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map='auto')
base_model = prepare_model_for_kbit_training(base_model)

lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
                      lora_dropout=0.05, target_modules=['c_attn'], fan_in_fan_out=True)

ADAPTER_DIR = 'adapters/trinity_e6'
NEED_TRAIN  = not os.path.exists(ADAPTER_DIR)
if NEED_TRAIN:
    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()
else:
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    print(f"♻️ 기존 E6 어댑터 재사용 (학습 생략)")

## 4. 질문 부분 손실 마스킹 + 학습 (~30분)

지금까지는 "질문+답변" 전체를 따라 쓰도록 학습했지만, 이제 질문 부분의 정답표(label)를 `-100`으로 막아
**모델이 답변 생성에만 학습 능력을 쓰도록** 합니다. `-100`은 "이 위치는 채점하지 말 것"이라는 약속된 값입니다
(수업 노트북 Q9에서 다룬 원본 KoChatGPT의 기법과 동일).

In [ ]:
from transformers import Trainer, TrainingArguments
from torch.nn.utils.rnn import pad_sequence

MAX_LEN = 256
PROMPT_TMPL = "### 질문: {q}\n\n### 답변:"

class MaskedSFTDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.items = []
        for d in data:
            src  = PROMPT_TMPL.format(q=d['prompt']) + ' '
            full = src + d['completion'] + tokenizer.eos_token
            ids  = tokenizer(full, truncation=True, max_length=MAX_LEN)['input_ids']
            n_src = len(tokenizer(src)['input_ids'])
            labels = list(ids)
            labels[:min(n_src, len(labels))] = [-100] * min(n_src, len(labels))  # 질문 부분 채점 제외
            self.items.append({'input_ids': ids, 'labels': labels})
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def masked_collator(batch):
    ids    = [torch.tensor(b['input_ids']) for b in batch]
    labels = [torch.tensor(b['labels'])    for b in batch]
    input_ids = pad_sequence(ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels    = pad_sequence(labels, batch_first=True, padding_value=-100)
    return {'input_ids': input_ids, 'labels': labels,
            'attention_mask': input_ids.ne(tokenizer.pad_token_id)}

train_min = 0.0
if NEED_TRAIN:
    train_ds = MaskedSFTDataset(train_set)
    args = TrainingArguments(
        output_dir='tmp_out', num_train_epochs=3,
        per_device_train_batch_size=4, gradient_accumulation_steps=8,
        learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type='cosine',
        fp16=True, logging_steps=50, gradient_checkpointing=True,
        save_strategy='no', report_to='none', seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=masked_collator)
    t0 = time.time()
    trainer.train()
    train_min = (time.time() - t0) / 60
    model.save_pretrained(ADAPTER_DIR)
    print(f"\n✅ 학습 완료 {train_min:.1f}분")

## 5. 메트릭 & 생성 함수 (E0~E5와 동일 — 채점 기준 불변)

In [ ]:
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smooth = SmoothingFunction().method1

def char_tokens(t): return list(t.replace(' ', ''))
def ngrams(toks, n): return Counter(tuple(toks[i:i+n]) for i in range(len(toks)-n+1))

def rouge_n(ref, hyp, n):
    r, h = ngrams(char_tokens(ref), n), ngrams(char_tokens(hyp), n)
    if not r or not h: return 0.0
    ov = sum((r & h).values())
    p, rec = ov / max(sum(h.values()), 1), ov / max(sum(r.values()), 1)
    return 2*p*rec/(p+rec) if p+rec else 0.0

def rouge_l(ref, hyp):
    a, b = char_tokens(ref), char_tokens(hyp)
    if not a or not b: return 0.0
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            dp[i][j] = dp[i-1][j-1]+1 if a[i-1]==b[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[-1][-1]; p, rec = lcs/len(b), lcs/len(a)
    return 2*p*rec/(p+rec) if p+rec else 0.0

def bleu(ref, hyp):
    r, h = char_tokens(ref), char_tokens(hyp)
    return sentence_bleu([r], h, weights=(0.5, 0.5), smoothing_function=smooth) if h else 0.0

def distinct_n(texts, n=2):
    allg = [g for t in texts for g in
            [tuple(char_tokens(t)[i:i+n]) for i in range(len(char_tokens(t))-n+1)]]
    return len(set(allg)) / max(len(allg), 1)

def score_set(refs, hyps):
    return {'BLEU'   : float(np.mean([bleu(r, h)       for r, h in zip(refs, hyps)])),
            'ROUGE-1': float(np.mean([rouge_n(r, h, 1) for r, h in zip(refs, hyps)])),
            'ROUGE-2': float(np.mean([rouge_n(r, h, 2) for r, h in zip(refs, hyps)])),
            'ROUGE-L': float(np.mean([rouge_l(r, h)    for r, h in zip(refs, hyps)])),
            'Distinct-2': float(distinct_n(hyps))}

@torch.no_grad()
def generate_batch(prompts, batch_size=4, max_new_tokens=96, min_new_tokens=12, **gen_kwargs):
    model.eval()
    tokenizer.padding_side = 'left'
    outs = []
    for i in range(0, len(prompts), batch_size):
        batch = [PROMPT_TMPL.format(q=p) for p in prompts[i:i+batch_size]]
        enc = tokenizer(batch, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_LEN).to(model.device)
        g = model.generate(**enc, max_new_tokens=max_new_tokens, min_new_tokens=min_new_tokens,
                           eos_token_id=tokenizer.eos_token_id,
                           pad_token_id=tokenizer.pad_token_id, **gen_kwargs)
        outs += [tokenizer.decode(g[j][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
                 for j in range(len(batch))]
    tokenizer.padding_side = 'right'
    return outs

print('준비 완료 | 참조 평균 길이', ref_avg_len, '자 → max_new_tokens=96, min_new_tokens=12로 길이 보정')

## 6. 디코딩 선발전 — 검증셋 20개로만 (테스트셋 미사용 ⚖️)

greedy / 빔서치 / 저온도 샘플링 세 후보를 검증셋에서 겨루게 하고,
루즈 조교님(ROUGE-L)과 블루 조교님(BLEU) 점수의 합이 가장 높은 후보만 테스트에 투입합니다.

In [ ]:
val_prompts = [d['prompt'] for d in val_set]
val_refs    = [d['completion'] for d in val_set]

CANDIDATES = {
    'greedy'  : dict(do_sample=False, repetition_penalty=1.2),
    'beam4'   : dict(do_sample=False, num_beams=4, no_repeat_ngram_size=2, repetition_penalty=1.2),
    'samp_t07': dict(do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.2),
}

t0 = time.time()
tournament = {}
for name, kw in CANDIDATES.items():
    hyps = generate_batch(val_prompts, **kw)
    s = score_set(val_refs, hyps)
    tournament[name] = (s['ROUGE-L'] + s['BLEU'], kw)
    print(f"{name:<9} BLEU {s['BLEU']:.4f} | ROUGE-L {s['ROUGE-L']:.4f} | 합계 {tournament[name][0]:.4f}")

winner = max(tournament, key=lambda k: tournament[k][0])
BEST_KW = tournament[winner][1]
print(f"\n🏆 선발전 우승: {winner} ({(time.time()-t0)/60:.1f}분)")

## 7. 최종 평가 — E0~E5와 동일한 테스트셋 60개로 단 한 번

In [ ]:
import pandas as pd

t0 = time.time()
hyps_e6 = generate_batch(test_prompts, **BEST_KW)
scores  = score_set(test_refs, hyps_e6)
print(f"생성 완료 ({(time.time()-t0)/60:.1f}분)\n")

row = {'exp': 'E6', 'desc': f'최종: 1.2B + 마스킹 + 강화필터 + 4k + {winner}',
       'model': 'trinity', 'clean': True, 'decode': winner,
       **{k: round(v, 4) for k, v in scores.items()}, 'train_min': round(train_min, 1)}

CSV = 'results/experiments.csv'
df = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame()
if len(df): df = df[df['exp'] != 'E6']
df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
df.to_csv(CSV, index=False)

with open('results/gen_E6.json', 'w', encoding='utf-8') as f:
    pyjson.dump({'prompts': test_prompts[:10], 'refs': test_refs[:10], 'hyps': hyps_e6[:10]},
                f, ensure_ascii=False, indent=1)

cols = ['exp', 'BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Distinct-2', 'train_min']
print(df[cols].to_string(index=False))

# 이전 결과가 있으면 비교 그래프
import matplotlib.pyplot as plt
order = [e for e in ['Base', 'E0', 'E1', 'E4', 'E5', 'E6'] if e in df['exp'].values]
if len(order) > 1:
    dfo = df.set_index('exp').loc[order].reset_index()
    metrics = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']
    x = np.arange(len(metrics)); w = 0.8 / len(dfo)
    colors = {'Base':'#999999','E0':'#4C72B0','E1':'#DD8452','E4':'#55A868','E5':'#C44E52','E6':'#8172B3'}
    plt.figure(figsize=(9, 4))
    for i, (_, r) in enumerate(dfo.iterrows()):
        plt.bar(x + i*w, [r[m] for m in metrics], w, label=r['exp'], color=colors.get(r['exp']))
    plt.xticks(x + w*(len(dfo)-1)/2, metrics); plt.legend()
    plt.title('Final comparison incl. E6'); plt.tight_layout(); plt.show()

## 8. 데모 — E0/E5와 같은 질문으로 정성 비교

In [ ]:
demo_questions = [
    "불고기용 고기 한우에요?",
    "리처드 닉슨이 43대 부통령직을 수행한 년도는?",
    "시카고 오헤어 국제공항은 어디에 있어?",
    "오늘 미세먼지 어때?",
]
demo_out = generate_batch(demo_questions, max_new_tokens=96, **BEST_KW)
with open('results/demo_E6.json', 'w', encoding='utf-8') as f:
    pyjson.dump(dict(zip(demo_questions, demo_out)), f, ensure_ascii=False, indent=1)

for q, a in zip(demo_questions, demo_out):
    print('='*70); print(f"❓ {q}"); print(f"  [E6] {a}")

print(f"\n⏱️ 전체 소요: {(time.time()-RUN_START)/60:.1f}분")
print("\n💾 백업: 아래 셀에서 BACKUP=True로 바꿔 실행하면 zip 다운로드")

In [ ]:
BACKUP = False
if BACKUP:
    !zip -qr ablation_backup_e6.zip results adapters
    from google.colab import files
    files.download('ablation_backup_e6.zip')

## 해석 포인트

- **E6 vs E5**: 같은 1.2B에서 마스킹+강화필터+증량+디코딩 선발이 얹은 추가 개선분 — "모델을 못 바꿀 때 데이터·학습 기법으로 얼마나 더 갈 수 있나"의 답
- **회피 응답 감소 확인**: `results/gen_E6.json`에서 "저는 AI~" 오프닝 빈도를 E5(6/10)와 비교해보세요
- **주의**: 검증셋(20개)으로 디코딩을 골랐기 때문에 테스트 점수는 오염 없는 공정한 수치입니다. 보고서에 "테스트셋 미접촉 원칙"을 명시하면 좋습니다
